# Lekcija 13 - Pomnilnik agenta


## Namestitev

Ta zvezek prikazuje, kako zgraditi agenta za rezervacijo potovanj z **trajnim spominom** z uporabo **Microsoft Agent Framework** (MAF).

Naučili se boste, kako različne vrste agentovega spomina — tekoči, kratkoročni in dolgoročni — vplivajo na to, kako agent ohranja in uporablja informacije skozi pogovore.

**Pogoji za začetek:**
- Projekt Azure AI Foundry z nameščenim modelom za klepet (npr. `gpt-4o-mini`).
- Prijava z Azure CLI — zaženite `az login` v svojem terminalu.
- `AZURE_AI_PROJECT_ENDPOINT` — konec vaše Azure AI Foundry projektne točke.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašega nameščenega modela.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Vrste pomnilnika agenta

AI agenti lahko uporabljajo različne vrste pomnilnika, od katerih ima vsaka različne namene:

### Delovni pomnilnik
Sam pogovor — sporočila, izmenjana v eni seji. Agent se lahko sklicuje na prejšnja sporočila v istem pogovoru, da ohrani koherenco. V MAF je to ustvarjeno prek **`agent.create_session()`**, kar vrne `AgentSession`.

### Kratkoročni pomnilnik
Informacije, ki obstajajo med izvajanjem naloge ali sejo, vendar niso trajno shranjene. Na primer, agent lahko med večkrožnim načrtovanjem zbira dejstva in jih uporablja za izdelavo končnega načrta.

### Dolgoročni pomnilnik
Nastavitve in dejstva, ki obstajajo **čez več sej**. Uporabnik, ki se vrne, ne bi smel ponavljati svojih prehranskih omejitev ali stila potovanja. Dolgoročni pomnilnik je običajno podprt z zunanjim shranjevanjem — podatkovno bazo, datoteko ali vektorskim indeksom — in agentu na voljo prek orodij.


## Delovni spomin z sejami

Najpreprostejša oblika spomina je seansa pogovora. Ko posredujete isto seanso (ustvarjeno prek `agent.create_session()`) zaporednim klicem `agent.run()`, agent vidi celotno zgodovino tega pogovora in se lahko spomni prej omenjenih podrobnosti.

Ustvarimo potovalnega agenta in pokažimo delovni spomin.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent si je pravilno zapomnil proračun, ker imata obe sporočili isti sejo. To je **delovni spomin** — obstaja samo med trajanjem seje.

### Kaj se zgodi z novo nitjo?

Če ustvarimo **novo** sejo, agent nima spomina na prejšnji pogovor:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Vzorec dolgoročnega spomina

Za zapomnitev uporabniških nastavitev **prek sej** potrebujemo trajen prostor za shranjevanje, ki deluje zunaj pogovorne niti. Agent do tega prostora dostopa prek **orodij** — funkcij, ki jih lahko kliče za shranjevanje in pridobivanje informacij.

Spodaj implementiramo preprost v spomin shranjeni prostor za preference (v produkciji bi to podprli z bazo podatkov ali vektorskim indeksom) in ga izpostavimo kot orodja, ki jih agent lahko uporablja.

### Arhitektura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Uporabnik, ki prvič uporablja, rezervira potovanje ob obletnici

Sarah obišče prvič. Agent naj shrani njene preference preko orodij in jih uporabi za priporočanje hotelov.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenarij 2 — Sarah se vrne tedne pozneje

Sarah začne **popolnoma novo nit** (simulacija nove seje). Delovni pomnilnik je prazen, vendar hranilnik dolgoročnih nastavitev še vedno vsebuje njene podatke. Agent jih mora pridobiti in uporabiti za personalizacijo priporočil.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Povzetek

V tej lekciji ste se naučili treh vrst agentovega spomina in kako jih uresničiti z Microsoft Agent Framework:

| Vrsta spomina | MAF mehanizem | Življenjska doba |
|---|---|---|
| **Delovni** | `agent.create_session()` | Ena sama konverzacija |
| **Kratkoročni** | Nakopičena vsebina konteksta znotraj teme | Ena naloga / seja |
| **Dolgoročni** | Zunanji pomnilnik dostopen preko funkcij `@tool` | Čez več sej |

### Ključne ugotovitve
1. **`agent.create_session()`** zagotavlja delovni pomnilnik — agent vidi celotno zgodovino pogovora znotraj seje.
2. **Nove seje izgubijo kontekst** — brez dolgoročnega spomina agent ne more priklicati prejšnjih pogovorov.
3. **Funkcije `@tool`** premostijo vrzel — omogočajo agentu shranjevanje in pridobivanje informacij iz trajnega shramba.
4. **Personalizacija se izboljšuje tekom časa** — več kot je shranjenih preferenc, boljše so agentove priporočila.

### Praktične uporabe
- **Storitve za stranke**: Pomnjenje zgodovine in preferenc strank
- **Osebni asistenti**: Ohranjanje konteksta čez dneve ali tedne
- **Zdravstvo**: Spremljanje informacij in preferenc pacientov
- **E-trgovina**: Personalizirano nakupovanje na podlagi zgodovine

### Naslednji koraki
- Zamenjati slovar v pomnilniku z bazo podatkov ali vektorsko zbirko (npr. Azure AI Search)
- Dodati potek roka spomina za časovno občutljive informacije
- Izgraditi sisteme z več agenti s skupnim spominom
- Raziščite Cognee zvezek za spomin, podprt z grafom znanja


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, prosimo, upoštevajte, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v izvorni jezik velja za avtoritativni vir. Za kritične informacije priporočamo strokovni človeški prevod. Nismo odgovorni za kakršne koli nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
